In [0]:
# ============================================================
# Notebook : 07_gold_moving_average
# Layer    : Gold
# Project  : Real-Time Stock Market Pipeline
#
# Purpose:
# Calculate 5, 10 and 20 period Simple Moving Averages
# from 1-minute OHLC candles.
#
# Input :
#   Silver OHLC Table
#
# Output :
#   Gold Moving Average Table
#
# Author : Shivam Mehta
# ============================================================

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

In [0]:
ohlc_df = spark.table("realtimedata.gold.ohlc_1min")

In [0]:
print("=" * 60)
print("Input Schema")
print("=" * 60)

ohlc_df.printSchema()

print("=" * 60)
print("Sample Data")
print("=" * 60)

display(ohlc_df)

Input Schema
root
 |-- symbol: string (nullable = true)
 |-- candle_time: timestamp (nullable = true)
 |-- open: double (nullable = true)
 |-- high: double (nullable = true)
 |-- low: double (nullable = true)
 |-- close: double (nullable = true)

Sample Data


symbol,candle_time,open,high,low,close
HDFCBANK,2026-06-29T10:38:00.000Z,798.9,798.9,798.9,798.9
HDFCBANK,2026-06-30T08:40:00.000Z,798.0,798.25,797.75,797.75
HDFCBANK,2026-06-30T08:41:00.000Z,797.6,798.15,797.5,797.75
HDFCBANK,2026-06-30T08:42:00.000Z,797.85,798.25,797.7,798.05
HDFCBANK,2026-06-30T08:43:00.000Z,798.05,798.25,798.0,798.0
HDFCBANK,2026-06-30T09:48:00.000Z,798.0,798.0,797.65,797.65
HDFCBANK,2026-06-30T09:49:00.000Z,797.65,798.2,797.6,798.2
HDFCBANK,2026-06-30T09:50:00.000Z,798.1,798.45,797.95,798.1
HDFCBANK,2026-06-30T09:51:00.000Z,798.1,798.1,797.6,798.0
HDFCBANK,2026-06-30T09:52:00.000Z,798.0,799.0,797.9,798.9


In [0]:
window_spec = (
    Window.partitionBy("symbol").orderBy(F.col("candle_time"))
)

In [0]:
window_5 = window_spec.rowsBetween(-4, 0)

window_10 = window_spec.rowsBetween(-9, 0)

window_20 = window_spec.rowsBetween(-19, 0)

In [0]:
moving_avg_df = (
    ohlc_df

    .withColumn(
        "sma_5",
        F.round(
            F.avg("close").over(window_5),
            2
        )
    )

    .withColumn(
        "sma_10",
        F.round(
            F.avg("close").over(window_10),
            2
        )
    )

    .withColumn(
        "sma_20",
        F.round(
            F.avg("close").over(window_20),
            2
        )
    )
)

In [0]:
print("=" * 60)
print("Moving Average Result")
print("=" * 60)

display(
    moving_avg_df.orderBy(
        "symbol",
        "candle_time"
    )
)

Moving Average Result


symbol,candle_time,open,high,low,close,sma_5,sma_10,sma_20
HDFCBANK,2026-06-29T10:38:00.000Z,798.9,798.9,798.9,798.9,798.9,798.9,798.9
HDFCBANK,2026-06-30T08:40:00.000Z,798.0,798.25,797.75,797.75,798.33,798.33,798.33
HDFCBANK,2026-06-30T08:41:00.000Z,797.6,798.15,797.5,797.75,798.13,798.13,798.13
HDFCBANK,2026-06-30T08:42:00.000Z,797.85,798.25,797.7,798.05,798.11,798.11,798.11
HDFCBANK,2026-06-30T08:43:00.000Z,798.05,798.25,798.0,798.0,798.09,798.09,798.09
HDFCBANK,2026-06-30T09:48:00.000Z,798.0,798.0,797.65,797.65,797.84,798.02,798.02
HDFCBANK,2026-06-30T09:49:00.000Z,797.65,798.2,797.6,798.2,797.93,798.04,798.04
HDFCBANK,2026-06-30T09:50:00.000Z,798.1,798.45,797.95,798.1,798.0,798.05,798.05
HDFCBANK,2026-06-30T09:51:00.000Z,798.1,798.1,797.6,798.0,797.99,798.04,798.04
HDFCBANK,2026-06-30T09:52:00.000Z,798.0,799.0,797.9,798.9,798.17,798.13,798.13


In [0]:
moving_avg_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("realtimedata.gold.moving_average")